In [1]:
import io
import pandas as pd
import requests
import os
import numpy as np
import matplotlib.pyplot as plt

url = "https://mydataaijournal.com/public/andy-data/olist-data/sla_breached.parquet"

print("Fetching dataset from mydataaijournal.com...")
response = requests.get(url)
response.raise_for_status()

df_sla = pd.read_parquet(io.BytesIO(response.content))
print(f"Successfully loaded {len(df_sla):,} rows into memory!")

Fetching dataset from mydataaijournal.com...
Successfully loaded 96,353 rows into memory!


In [2]:
df_sla.columns = df_sla.columns.str.lower()

In [3]:
df_sla.head()

,order_id,sla_delay_days,is_sla_breached,review_score
0,73fc7af87114b39712e6da79b0a377eb,-16,0,4
1,a548910a1c6147796b98fdf73dbeba33,-5,0,5
2,f9e4b658b201a9f2ecdecbb34bed034b,-21,0,5
3,658677c97b385a9be170737859d3511b,-20,0,5
4,8e6bfb81e283fa7e4f11123a3fb894f1,-9,0,5


In [4]:
x_real = df_sla["sla_delay_days"]
y_real = df_sla["review_score"]



<div style="width:75%;margin:auto;font-size:1.2em;">

### The Research Question

**"Is there a statistically significant linear relationship between the number of days a delivery is delayed beyond its estimated SLA and the resulting customer review score?"**

*Can we mathematically prove that as delivery delay increases, customer satisfaction systematically changes?*

* An SLA (Service Level Agreement) is simply the promised delivery deadline given to a customer when they place an order.*

### Simple Example: 

* **Promised Deadline:** Customer orders an item and the system estimates **"Delivers by Sep 15"**.
* **On Time (SLA Met):** Package arrives on or before Sep 15 $\rightarrow$ `sla_delay_days <= 0`.
* **Late (SLA Breached):** Package arrives on Sep 18 $\rightarrow$ Breached by 3 days (`sla_delay_days = 3`).

---

### Hypothesis Statements

In simple linear regression, hypothesis testing focuses on the **slope parameter ($\beta_1$)**. If the slope is zero, the independent variable ($X$) has no linear effect on the dependent variable ($Y$).

#### 1. The Null Hypothesis ($H_0$)

* There is no linear relationship between SLA delay days and customer review scores. Any observed slope in the data is purely due to random chance.
* **Mathematical Statement:** $H_0: \beta_1 = 0$

#### 2. The Alternative Hypothesis ($H_a$)

* There is a statistically significant linear relationship between SLA delay days and customer review scores. As delay days change, the review score changes.
* **Mathematical Statement:** $H_a: \beta_1 \neq 0$

</div>

In [5]:
import numpy as np
import pandas as pd
import scipy.stats as stats


def parametric_regression(x, y, alpha=0.05):
    # 1. Fit linear regression model
    res = stats.linregress(x, y)
    obs_slope = res.slope
    p_value = res.pvalue  # Two-tailed p-value based on Student's t-distribution
    std_err = res.stderr

    # 2. Compute 95% Confidence Interval for the slope
    n = len(x)
    df = n - 2  # Degrees of freedom for simple linear regression
    t_crit = stats.t.ppf(1 - alpha / 2, df=df)  # Two-tailed critical value

    me = margin_of_error = t_crit * std_err
    ci_lower = obs_slope - me
    ci_upper = obs_slope + me

    return p_value, obs_slope, std_err, (ci_lower, ci_upper)


# Run parametric regression
p_value, obs_slope, std_err, (ci_lower, ci_upper) = parametric_regression(
    x_real.values, y_real.values
)

print("--- Parametric Linear Regression Results ---")
print(f"Observed Slope (β1): {obs_slope:.5f}")
print(f"Standard Error:      {std_err:.5f}")
print(f"Parametric p-value:  {p_value:.5e}")
print(f"95% CI for Slope:    [{ci_lower:.5f}, {ci_upper:.5f}]")

--- Parametric Linear Regression Results ---
Observed Slope (β1): -0.03393
Standard Error:      0.00039
Parametric p-value:  0.00000e+00
95% CI for Slope:    [-0.03470, -0.03316]


<div style="width:80%;margin:auto;font-size:1.1em;">

### 📊 Conclusion & Business Impact

> **Research Question:** *Is there a statistically significant linear relationship between SLA delay days and customer review scores?*

#### 🎯 Statistical Decision

* Since p-value is well below the standard significance threshold ($\alpha = 0.05$), we reject the Null 
and accept the Alternative Hypothesis.
* There is a statistically significant negative linear relationship between SLA delay days and customer review scores.


</div>